In [ ]:
!pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6

In [ ]:
import os
import gradio as gr
from google.colab import drive,userdata
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,pipeline
import torch

In [ ]:
LLAMA = "meta-llama/Llama-3.2-3B-Instruct"

In [ ]:
drive.mount("/content/drive")
audio_filename = "/content/drive/MyDrive/llms/denver_extract.mp3"

In [ ]:
# Sign in to HuggingFace Hub

hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

# Open the file

audio_file = open(audio_filename, "rb")

In [ ]:


pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-medium.en",
    dtype=torch.float16,
    device="cuda",
    return_timestamps=True
)

def transcribe_audio(audio_file):
    if audio_file is None:
        return "Please upload an audio file."

    result = pipe(audio_file)
    return result["text"]

In [ ]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

In [ ]:
def meeting_notes(transcription, uploaded, drive, language):

    if not transcription:
        return "Please generate the transcription first."

    selected_file = uploaded if uploaded else drive

    if not selected_file:
        return "Please upload or select an audio file."

    system_message = f"""
    You produce minutes of meetings from transcripts.
    Generate the meeting notes in {language}.
    Include summary, key discussion points, takeaways,
    and action items with owners.
    Use markdown format without code blocks.
    """

    user_prompt = f"""
    Below is an English transcript of a Denver council meeting.

    Please write the meeting minutes in {language}, including:
    - a summary with attendees, location and date
    - discussion points
    - takeaways
    - action items with owners

    Transcription:
    {transcription}
    """

    messages = [
        {
            "role": "system",
            "content": system_message
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]

    tokenizer = AutoTokenizer.from_pretrained(LLAMA)
    tokenizer.pad_token = tokenizer.eos_token

    inputs = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        add_generation_prompt=True
    ).to("cuda")

    model = AutoModelForCausalLM.from_pretrained(
        LLAMA,
        device_map="auto",
        quantization_config=quant_config
    )

    outputs = model.generate(
        inputs,
        max_new_tokens=2000
    )

    response = tokenizer.decode(
        outputs[0][inputs.shape[-1]:],
        skip_special_tokens=True
    )

    return response

In [ ]:

DRIVE_FOLDER = "/content/drive/MyDrive/llms"

audio_files = [
    os.path.join(DRIVE_FOLDER, f)
    for f in os.listdir(DRIVE_FOLDER)
    if f.endswith((".mp3", ".wav", ".m4a"))
]

with gr.Blocks() as demo:
    gr.Markdown("# Denver Council Meeting Notes")

    upload_file = gr.File(
        label="Upload Audio File",
        file_types=[".mp3", ".wav", ".m4a"],
        type="filepath"
    )

    drive_file = gr.Dropdown(
        choices=audio_files,
        label="Select audio file from Google Drive"
    )
  
    generate_button = gr.Button("Generate Transcription")

    notes_button = gr.Button("notes_button Transcription")

    transcription_output = gr.Textbox(
        label="Transcription",
        lines=15
    )

    meeting_notes_output = gr.Textbox(
        label="Transcription",
        lines=15
    )

    notes_language = gr.Dropdown(
      choices=["English", "Marathi", "Hindi", "Spanish", "French"],
      label="Meeting Notes Language",
      value="Spanish"
    )

    meeting_notes_format = gr.Markdown(
      label="Meeting Notes"
    )

    def generate_transcription(uploaded, drive):
      selected_file = uploaded if uploaded else drive

      if not selected_file:
          return "Please upload or select an audio file."

      result = pipe(selected_file)
      return result["text"]
    
    generate_button.click(
        fn=generate_transcription,
        inputs=[upload_file, drive_file],
        outputs=transcription_output
    )

    notes_button.click(
      fn=meeting_notes,
      inputs=[
        transcription_output,
        upload_file,
        drive_file,
        notes_language
    ],
      outputs=meeting_notes_output
    )
   
    demo.launch(inbrowser=True)
